# Ewaluacja wytrenowanych modeli\n\nNotebook ładuje checkpointy zapisane przez `train_mlp.py`, `train_cnn.py` i `train_cnn_aug.py`, a potem liczy metryki na zbiorze testowym.

In [ ]:
from pathlib import Path\nimport sys\n\nROOT = Path.cwd()\nif ROOT.name == 'notebooks':\n    ROOT = ROOT.parent\nsys.path.insert(0, str(ROOT / 'src'))\n\nDATA_DIR = (ROOT.parent / 'data' / 'raw' / 'kaggle_room_street_data' / 'house_data').resolve()\nSPLIT_CSV = ROOT / 'splits' / 'room_split_seed42.csv'\nCHECKPOINT_DIR = ROOT / 'outputs' / 'checkpoints'\nREPORT_DIR = ROOT / 'outputs' / 'reports'\n\nprint('ROOT:', ROOT)\nprint('DATA_DIR:', DATA_DIR)\nprint('CHECKPOINT_DIR:', CHECKPOINT_DIR)

In [ ]:
import pandas as pd\nfrom room_classifier.data import load_or_create_split, split_summary\n\nsplit_frame = load_or_create_split(DATA_DIR, SPLIT_CSV, exclude_classes=['bath'], seed=42)\nsplit_summary(split_frame)

In [ ]:
checkpoints = sorted(CHECKPOINT_DIR.glob('*best.pt'))\ncheckpoints

In [ ]:
from room_classifier.evaluate import evaluate_checkpoint, plot_confusion_matrix\n\nresults = []\nfor checkpoint in checkpoints:\n    metrics = evaluate_checkpoint(\n        checkpoint_path=checkpoint,\n        data_dir=DATA_DIR,\n        split_csv=SPLIT_CSV,\n        split='test',\n        batch_size=64,\n        num_workers=0,\n        output_dir=REPORT_DIR,\n    )\n    results.append(metrics)\n\nsummary = pd.DataFrame([\n    {\n        'checkpoint': Path(m['checkpoint']).name,\n        'model_type': m['model_type'],\n        'arch': m['arch'],\n        'accuracy': m['accuracy'],\n        'macro_f1': m['macro_f1'],\n    }\n    for m in results\n]).sort_values(['macro_f1', 'accuracy'], ascending=False)\nsummary

In [ ]:
for metrics in results:\n    print('\\n', Path(metrics['checkpoint']).name)\n    display(pd.DataFrame(metrics['classification_report']).T.round(3))

In [ ]:
import matplotlib.pyplot as plt\n\nfor metrics in results:\n    fig, ax = plot_confusion_matrix(metrics['confusion_matrix'], metrics['class_names'])\n    ax.set_title(Path(metrics['checkpoint']).name)\n    plt.show()